<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment4/DNN_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import time

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train[..., np.newaxis] / 255.0, x_test[..., np.newaxis] / 255.0

# Create tf.data.Dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(64)

# Create a simple CNN model
# def create_model():
#     model = models.Sequential([
#         layers.Input(shape=(28, 28, 1)),
#         layers.Conv2D(32, 3, activation='relu'),
#         layers.MaxPooling2D(),
#         layers.Flatten(),
#         layers.Dense(128, activation='relu'),
#         layers.Dense(10)
#     ])
#     return model
def create_model():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Flatten(),                # Flatten the 28x28 image into a vector
        layers.Dense(8, activation='relu'),   # 1st hidden layer with 8 neurons
        layers.Dense(4, activation='relu'),   # 2nd hidden layer with 4 neurons
        layers.Dense(2, activation='relu'),   # 3rd hidden layer with 2 neurons
        layers.Dense(10)                      # Output layer with 10 classes (logits)
    ])
    return model


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
test_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

model_manual = create_model()
optimizer = tf.keras.optimizers.Adam()

# Training loop
EPOCHS = 5
start_time = time.time()
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            logits = model_manual(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        grads = tape.gradient(loss, model_manual.trainable_weights)
        optimizer.apply_gradients(zip(grads, model_manual.trainable_weights))
        train_acc_metric.update_state(y_batch, logits)

    print("Training Accuracy:", train_acc_metric.result().numpy())
    train_acc_metric.reset_state()

end_manual = time.time()



Epoch 1/5
Training Accuracy: 0.20581667

Epoch 2/5
Training Accuracy: 0.30768332

Epoch 3/5
Training Accuracy: 0.4914

Epoch 4/5
Training Accuracy: 0.6350167

Epoch 5/5
Training Accuracy: 0.75215


In [ ]:
model_fit = create_model()
model_fit.compile(optimizer='adam',
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=['accuracy'])

start_fit = time.time()
model_fit.fit(x_train, y_train, epochs=5, batch_size=64, verbose=2)
end_fit = time.time()


Epoch 1/5
938/938 - 5s - 6ms/step - accuracy: 0.2864 - loss: 1.7724
Epoch 2/5
938/938 - 2s - 2ms/step - accuracy: 0.4284 - loss: 1.3935
Epoch 3/5
938/938 - 3s - 3ms/step - accuracy: 0.6481 - loss: 1.0451
Epoch 4/5
938/938 - 2s - 2ms/step - accuracy: 0.7054 - loss: 0.8733
Epoch 5/5
938/938 - 2s - 2ms/step - accuracy: 0.7297 - loss: 0.7947


In [ ]:
# Reuse loss and metric objects
test_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
test_loss = tf.keras.metrics.Mean()

# Evaluate manually
for x_batch, y_batch in test_ds:
    logits = model_manual(x_batch, training=False)
    loss = loss_fn(y_batch, logits)
    test_loss.update_state(loss)
    test_acc_metric.update_state(y_batch, logits)

# Results
loss_manual = test_loss.result().numpy()
acc_manual = test_acc_metric.result().numpy()

loss_fit, acc_fit = model_fit.evaluate(x_test, y_test, verbose=0)

print("\n--- Performance Comparison ---")
print(f"Manual Training Accuracy : {acc_manual:.4f}")
print(f"model.fit() Accuracy     : {acc_fit:.4f}")
print(f"Manual Training Time     : {end_manual - start_time:.2f} sec")
print(f"model.fit() Time         : {end_fit - start_fit:.2f} sec")



--- Performance Comparison ---
Manual Training Accuracy : 0.7739
model.fit() Accuracy     : 0.7397
Manual Training Time     : 260.93 sec
model.fit() Time         : 15.60 sec
